# Notebook 11 – Logits and Probabilities

## Concept
A model does not output "Positive" or "Negative" directly. Its last layer outputs one raw number per class, called a **logit**. Logits can be any value (negative, positive, large, small) and do not add up to 1.

**Softmax** converts logits into **probabilities** (all between 0 and 1, and they add up to 1). This set of probabilities is the **probability distribution**. The **predicted class** is the class with the highest probability (`argmax`).

```
Model → Logits → Softmax → Probabilities → Predicted Class
```

## Why it's needed
- **Why models produce logits:** raw scores are easy to compute and train with; the model just says "how strongly do I favour each class".
- **Why softmax:** humans and applications need values that can be read as confidence and compared (e.g. 92% vs 5% vs 3%).
- **Why prediction selection:** we must finally pick one answer, so we take the class with the highest probability.

In [1]:
import numpy as np

classes = ["Negative", "Neutral", "Positive"]

def softmax(logits):
    exp = np.exp(logits - np.max(logits))   # subtract max for numerical stability
    return exp / exp.sum()

def predict(text, logits, true_label):
    probs = softmax(logits)                  # Logits -> Softmax -> Probabilities
    pred_idx = int(np.argmax(probs))         # Probabilities -> Predicted Class
    print("Text         :", text)
    print("Logits       :", np.round(logits, 2), "| sum =", round(float(logits.sum()), 2))
    print("Probabilities:", np.round(probs, 3), "| sum =", round(float(probs.sum()), 2))
    print("Predicted    :", classes[pred_idx], f"({probs[pred_idx]:.1%} confidence)")
    print("Actual       :", true_label)
    print("Result       :", "CORRECT" if classes[pred_idx] == true_label else "WRONG")
    print("-" * 60)

# Case 1: model output (logits) for a clear sentence
predict("I love this phone, it is amazing!",
        np.array([-1.2, 0.3, 2.5]), "Positive")

# Case 2: the model is very confident, but wrong
predict("The delivery was okay, nothing special.",
        np.array([-0.5, 0.4, 3.0]), "Neutral")

Text         : I love this phone, it is amazing!
Logits       : [-1.2  0.3  2.5] | sum = 1.6
Probabilities: [0.022 0.098 0.881] | sum = 1.0
Predicted    : Positive (88.1% confidence)
Actual       : Positive
Result       : CORRECT
------------------------------------------------------------
Text         : The delivery was okay, nothing special.
Logits       : [-0.5  0.4  3. ] | sum = 2.9
Probabilities: [0.027 0.067 0.905] | sum = 1.0
Predicted    : Positive (90.5% confidence)
Actual       : Neutral
Result       : WRONG
------------------------------------------------------------


## What this example tells us
- Logits are raw scores: they can be negative and do not sum to 1.
- Softmax turns them into probabilities that sum to 1; `argmax` picks the predicted class.
- In Case 2 the model gave about 90% to "Positive", yet the true label was "Neutral". **A high probability is confidence, not correctness.**

**Why the highest probability does not always mean the model is correct**
- Probability only shows how strongly the model *believes* its answer, based on what it learned from training data.
- Softmax always produces a confident-looking result, even for sentences the model has never seen or finds ambiguous.
- Models can be overconfident (poorly calibrated), and biased or noisy training data leads to confident mistakes.
- Always check accuracy on real labelled data, not just the confidence score.

## AI/ML Example
A Hugging Face sentiment model returns logits such as `[-1.2, 0.3, 2.5]` from `outputs.logits`. We apply `torch.softmax(outputs.logits, dim=-1)` and `argmax` to get the label. A sarcastic review like "Great, it broke in one day" may still get 95% "Positive" and be wrong.

## Business Example
A support-ticket classifier says "Not urgent" with 91% probability, but the customer is actually threatening to cancel. If the business auto-closes every ticket above 90% confidence, urgent cases are missed. Low-confidence or high-risk tickets should go to a human.

## AI/ML Engineer Use Case
Use probabilities as a signal, not a guarantee: set a confidence threshold, route low-confidence predictions to human review, measure calibration and accuracy on a labelled test set, and log wrong-but-confident predictions for failure analysis (see Notebook 15).